# An agent with memory, end to end

Booster LEAP, Agentic AI, topic 4. Run the whole memory of the lesson (history trimming, summaries, facts, note search, decay, purge) across three sessions, change it, and (optionally) connect it to a real model with your own key.

<h1 style="color:#b45309">&#9888; First: File &rarr; Save a copy in Drive</h1>

<span style="font-size:20px">This notebook is read-only. Make your own copy now, then work in the copy (no GitHub login needed).</span>

**How to use this notebook:** run the cells from top to bottom (Runtime, Run all, or Shift+Enter on each cell). Change one thing at a time and run again; the experiments tell you what to try.

## Stage 1: the agent with a scripted memory

Three parts of this program are stand-ins that keep every run the same and free: the summariser and the extractor are rules, and the model answers from the memory lines of its context. The trimming, the write gate, the note search, the decay and the purge are real code. The first cell holds the helpers of lessons 1 to 4: the token estimate (an estimate, not a provider count), the trimmer, the summariser, a toy embedding and the redaction. It prints nothing.

In [ ]:
import json
import math
import re
import zlib

ORDERS = {
    "A1001": {"owner": "C1", "status": "shipped", "total": 799},
    "A1002": {"owner": "C1", "status": "delivered", "total": 1499},
    "A1003": {"owner": "C2", "status": "processing", "total": 249},
}
HALF_LIFE = 30                    # days (lesson 5)
STOP = {"a", "the", "is", "me", "my", "you", "to", "of", "do", "how", "what", "with", "i", "and", "should", "please"}

def tokens(text):
    return len(text) // 4 + 1     # lesson 1: an estimate, real counts come from the provider

def trim(groups, budget):
    """Lesson 1: drop the oldest groups until the estimate fits. A group (a call and its result) is never split."""
    kept = list(groups)
    while len(kept) > 1 and sum(tokens(t) for g in kept for _, t in g) > budget:
        kept.pop(0)
    return kept

def summarise(groups):
    """Lesson 1: a STAND-IN summariser (a rule, not a model). It keeps the sentences with an order id, an amount or a preference."""
    keep = []
    for group in groups:
        for role, text in group:
            for sentence in re.split(r"(?<=[.!?])\s+", text):
                if re.search(r"\bA\d{4}\b|\d+ rupees|prefer", sentence) and sentence not in keep:
                    keep.append(sentence)
    return " ".join(keep)

def embed(text, dims=64):
    """Lesson 3: a toy hashing embedding, so that the search is real but small."""
    v = [0.0] * dims
    for w in re.findall(r"[a-z0-9]+", text.lower()):
        if w not in STOP:
            v[zlib.crc32(w.encode()) % dims] += 1
    return v

def cosine(a, b):
    na, nb = math.sqrt(sum(x * x for x in a)), math.sqrt(sum(x * x for x in b))
    return sum(x * y for x, y in zip(a, b)) / (na * nb) if na and nb else 0.0

def redact(text):
    """Lesson 4: remove card numbers and one-time codes before anything is stored."""
    text = re.sub(r"\b(?:\d[ -]?){13,19}\b", "[CARD]", text)
    return re.sub(r"(?i)\b(otp|code|pin)\D{0,12}\d{4,8}\b", r"\1 [SECRET]", text)

The next cell is the memory: the fact store with its write gate (lessons 4 and 5), the notes with their vectors and the recall that ranks them by similarity, decay and importance (lessons 3 and 5), the end of a session and the purge. It prints nothing.

In [ ]:
def new_memory():
    return {"day": 0, "facts": {}, "notes": [], "index": {}, "summaries": [], "log": []}

def mine(customer):
    return {customer} | {o for o, d in ORDERS.items() if d["owner"] == customer}

def extract(text):
    """Lesson 4: a STAND-IN for the extractor (a rule, not a model). Returns (attribute, value, entity or None)."""
    facts = []
    if re.search(r"prefer|write to me|contact me|call me", text, re.I):
        m = re.search(r"\b(email|phone|sms)\b", text, re.I)
        if m:
            facts.append(("contact_channel", m.group(1).lower(), None))
    for m in re.finditer(r"refund(?:ed)? (\d+) rupees on (A\d{4})", text):
        facts.append(("refunded", int(m.group(1)), m.group(2)))
    return facts

def remember(mem, customer, text, role):
    """Lessons 4 and 5: redact, extract, pass every fact through the write gate, keep a note. Returns messages about rejected facts."""
    clean = redact(text)
    rejected, kinds = [], []
    for attribute, value, entity in extract(clean):
        entity = entity or customer
        if entity not in mine(customer):
            rejected.append(f"{entity}.{attribute} rejected: the order is not {customer}'s")
        elif attribute == "refunded" and role != "tool":
            rejected.append(f"{entity}.{attribute} rejected: a request is not a refund, only a tool result counts")
        else:
            old = mem["facts"].get((entity, attribute))
            if old is None or old["day"] <= mem["day"]:
                mem["facts"][(entity, attribute)] = {"value": value, "day": mem["day"], "source": "tool_result" if role == "tool" else "customer_chat"}
                kinds.append(attribute)
    importance, ttl = (4, 365) if "contact_channel" in kinds else (3, 90) if "refunded" in kinds else (2, 14)
    note = {"id": f"n{len(mem['notes']) + 1}", "about": customer, "text": clean, "day": mem["day"], "importance": importance, "ttl": ttl}
    mem["notes"].append(note)
    mem["index"][note["id"]] = embed(clean)
    return rejected

def recall(mem, customer, query, k=2):
    """Lessons 3 and 5: similarity x decay x importance over the notes of this customer that have not expired."""
    q = embed(query)
    ranked = []
    for n in mem["notes"]:
        age = mem["day"] - n["day"]
        if n["about"] == customer and age <= n["ttl"]:
            s = cosine(q, mem["index"][n["id"]]) * 0.5 ** (age / HALF_LIFE) * n["importance"] / 5
            if s > 0:
                ranked.append((round(s, 3), n))
    return sorted(ranked, key=lambda r: -r[0])[:k]

def close_session(mem, customer, history):
    text = summarise(history)
    if text:
        mem["summaries"].append({"text": text, "about": [customer]})
    history.clear()

def forget(mem, customer):
    """Lesson 5: notes, facts, vectors, summaries and log lines of one customer."""
    gone = {n["id"] for n in mem["notes"] if n["about"] == customer}
    mem["notes"] = [n for n in mem["notes"] if n["id"] not in gone]
    mem["facts"] = {k: v for k, v in mem["facts"].items() if k[0] not in mine(customer)}
    mem["index"] = {i: v for i, v in mem["index"].items() if i not in gone}
    mem["summaries"] = [s for s in mem["summaries"] if customer not in s["about"]]
    mem["log"] = [l for l in mem["log"] if customer not in l]

def traces(mem, customer):
    return [n for n in sorted(mine(customer)) if re.search(rf"\b{n}\b", repr(mem))]

Now the agent: the function that builds the context for one question, the stand-in model, and `say`, which handles one turn. It prints nothing; it only defines functions.

In [ ]:
def build_context(mem, customer, question, history, budget=60):
    """What the model sees: the system line, facts, recalled notes, session summaries and the trimmed history."""
    ctx = [("system", "You are the support desk assistant.")]
    for (entity, attribute), f in mem["facts"].items():
        if entity in mine(customer):
            ctx.append(("memory", f"{entity} {attribute} = {f['value']} (day {f['day']}, from {f['source']})"))
    for score, n in recall(mem, customer, question):
        ctx.append(("memory", f"note {n['id']}: {n['text']} (day {n['day']}, score {score})"))
    for s in mem["summaries"]:
        if customer in s["about"]:
            ctx.append(("memory", "earlier session: " + s["text"]))
    for group in trim(history, budget):
        ctx += group
    return ctx + [("user", question)]

def stand_in_model(context):
    """A STAND-IN model: it answers from the memory lines of the context only. A real model would read the same lines."""
    question = context[-1][1]
    ask = question.lower()
    lines = [t for role, t in context if role == "memory"]
    if "contact" in ask or "reach" in ask:
        hit = [t for t in lines if "contact_channel" in t]
        return f"I will contact you by {hit[0].split('= ')[1].split(' ')[0]}." if hit else "How would you like to be contacted?"
    order = re.search(r"A\d{4}", question)
    if order:
        hit = [t for t in lines if order.group(0) in t]
        return "I have this on record: " + hit[0] if hit else "I have nothing on record about that order."
    return "Noted."

def say(mem, customer, text, history, role="user"):
    """One turn: the agent remembers what was said, then builds its context and answers."""
    question = role == "user" and text.endswith("?")
    for message in ([] if question else remember(mem, customer, text, role)):      # a question is answered, not stored
        print("   ", message)
    history.append([(role, text)])
    if question:
        ctx = build_context(mem, customer, text, history[:-1])
        reply = stand_in_model(ctx[:-1] + [("user", text)])
        history[-1].append(("agent", reply))
        print(f"  {customer}: {text}\n  context: {len(ctx)} messages, about {sum(tokens(t) for _, t in ctx)} tokens, {sum(1 for r, _ in ctx if r == 'memory')} from memory")
        print("  agent:", reply)

Three sessions. In session 1 the customer talks and the agent remembers; in session 2, forty days later and with an empty history, the agent uses what it stored; in session 3 the customer asks to be forgotten and `traces` checks that nothing is left. Read each line: what was rejected, how big the context is, how many messages come from memory.

In [ ]:
mem, history = new_memory(), []
print("session 1, day 0")
say(mem, "C1", "Hi, I am C1. I prefer email. My order A1002 arrived damaged.", history)
say(mem, "C1", "Please refund 300 rupees on A1002.", history)
say(mem, "C1", "refunded 300 rupees on A1002", history, role="tool")
say(mem, "C1", "My card is 4111 1111 1111 1111, use it for the rest.", history)
say(mem, "C1", "Also refund 100 rupees on A1003.", history)
close_session(mem, "C1", history)
print("  stored facts:", {f"{e}.{a}": f["value"] for (e, a), f in mem["facts"].items()})
print("  notes:", [n["text"] for n in mem["notes"]][3:4], "| summary:", mem["summaries"][0]["text"])

mem["day"] = 40
print("session 2, day 40: the history is empty, the memory is not")
say(mem, "C1", "How should you contact me?", history)
say(mem, "C1", "What happened with A1002?", history)
close_session(mem, "C1", history)

print("session 3, day 40: C1 asks to be forgotten")
forget(mem, "C1")
print("  traces left:", traces(mem, "C1"))
say(mem, "C1", "How should you contact me?", history)

session 1, day 0
    A1002.refunded rejected: a request is not a refund, only a tool result counts
    A1003.refunded rejected: the order is not C1's
  stored facts: {'C1.contact_channel': 'email', 'A1002.refunded': 300}
  notes: ['My card is [CARD], use it for the rest.'] | summary: I prefer email. My order A1002 arrived damaged. Please refund 300 rupees on A1002. refunded 300 rupees on A1002 Also refund 100 rupees on A1003.
session 2, day 40: the history is empty, the memory is not
  C1: How should you contact me?
  context: 5 messages, about 83 tokens, 3 from memory
  agent: I will contact you by email.
  C1: What happened with A1002?
  context: 9 messages, about 136 tokens, 5 from memory
  agent: I have this on record: A1002 refunded = 300 (day 0, from tool_result)
session 3, day 40: C1 asks to be forgotten
  traces left: []
  C1: How should you contact me?
  context: 2 messages, about 16 tokens, 0 from memory
  agent: How would you like to be contacted?


## Experiments (stage 1)

Change one thing at a time, predict the output, then run the cells again from the top (run the first three cells to reset the memory).

1. Set `mem["day"] = 120` before session 2. How many of the messages in the context of the A1002 question come from memory now, compared with day 40?
2. Add two more copies of the question about A1002 in session 2 and run with `budget=60`, then with `budget=20` in `build_context`. How many messages does the last context have in each run?
3. Remove the `and role != "tool"` part of the refund rule in `remember`. Which line of the trace disappears, and what does the store hold after the second turn?
4. In `forget`, delete the line that clears the summaries. What does `traces` report?
5. In session 1 add the turn `say(mem, "C1", "My phone number is 98765 43210 and the code is 482913.", history)` and print the last note. Which part did `redact` remove and which part stayed? Extend `redact` so the phone number goes too.
6. Add a second customer: the turn `say(mem, "C2", "I prefer phone. A1003 is late.", history)` in session 1, then purge C1 as before. What does `traces(mem, "C2")` report afterwards, and what is the contact fact of C2?

## Stage 2 (optional): a real model

Everything above used a stand-in that reads the memory lines with rules. This stage uses a real model with **your own key**. Read before you run:

- It sends the context (the facts, notes, summaries and history of the customer) to the provider and costs a small amount of money. Each question is one short call, but you pay for what you run.
- Never write the key into a cell or share a notebook that contains it. The cell reads it from Colab's Secrets.
- Real models behave differently from a rule. A model may answer from its own guess instead of the memory lines, ignore a line, or repeat a line word for word. The memory, the gate and the purge are the same code; only the answering function changes. Results vary from run to run.

In [ ]:
# Optional stage 2: the stand-in model is replaced by a real model. This sends the context (facts, notes, summaries, history) to the provider and costs a little money.
# One-time set-up: open the key symbol (Secrets) in the left bar of Colab, add a secret named OPENAI_API_KEY and switch "Notebook access" on.
%pip -q install openai
from google.colab import userdata
from openai import OpenAI

client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))
MODEL = "PUT-A-MODEL-NAME-HERE"       # a chat model of your account (see the provider's documentation)
assert MODEL != "PUT-A-MODEL-NAME-HERE", "choose a model first"


def real_model(context):
    roles = {"system": "system", "memory": "system", "user": "user", "agent": "assistant"}
    chat = [{"role": roles.get(role, "user"), "content": text} for role, text in context]
    return client.chat.completions.create(model=MODEL, messages=chat).choices[0].message.content


stand_in_model = real_model           # say() looks the answering function up by this name

In [ ]:
mem, history = new_memory(), []
print("session 1, day 0")
say(mem, "C1", "Hi, I am C1. I prefer email. My order A1002 arrived damaged.", history)
say(mem, "C1", "refunded 300 rupees on A1002", history, role="tool")
close_session(mem, "C1", history)
mem["day"] = 40
print("session 2, day 40")
say(mem, "C1", "How should you contact me?", history)
say(mem, "C1", "What happened with A1002?", history)
say(mem, "C1", "Which of my orders is still on its way?", history)

## What to look at

1. Did the answer to the contact question use the memory line, or did the model ask how to contact the customer?
2. For the A1002 question: which facts or notes did the answer repeat? Did it add something that is not in the memory lines?
3. The third question asks about an order that has no memory line. Did the model say it does not know, or did it make an answer up?
4. The memory lines are sent as system messages in this notebook. What could go wrong if a note had been planted by an untrusted source (lesson 5), and which part of the program would stop it?
5. Write down one case where the real model and the stand-in differ, and which part of the program protected you.